# Notebook 2: Selecting, Filtering, and Renaming Columns

## Goal

The goal of this notebook is to learn the core Spark DataFrame operations you will use constantly.

These operations are the foundation for most PySpark workflows.

You will learn:

| Topic | Practice |
|---|---|
| Select columns | `select()` |
| Filter rows | `filter()`, `where()` |
| Rename columns | `withColumnRenamed()` |
| Create new columns | `withColumn()` |
| Drop columns | `drop()` |
| Sort data | `orderBy()` |
| Limit rows | `limit()` |

## Mini-Project

From the retail transaction data, you will:

1. Filter for high-value transactions
2. Select important columns
3. Rename columns cleanly
4. Create derived columns
5. Sort by transaction amount

## Key Lesson

Most Spark workflows start with a simple pattern:

```text
Start with a DataFrame
   ↓
Select useful columns
   ↓
Filter relevant rows
   ↓
Create or rename columns
   ↓
Sort or limit results
```
These operations are transformations. Spark does not execute them until you call an action such as display(), show(), or count().

In [0]:
from pyspark.sql import functions as F

from pyspark.sql.types import (
   StructType,
   StructField,
   IntegerType,
   StringType,
   DoubleType,
   TimestampType
)

from datetime import datetime


In [0]:
# Confirm SparkSession exists.
# In Databricks, spark is usually available automatically.

spark

In [0]:
print("Spark version:", spark.version)

Spark version: 4.1.0


# Section 1: Create Retail Transaction Data

## 1. Create a Retail Transaction DataFrame

We will create a small retail transaction dataset.

The grain of this DataFrame is:

> One row per transaction.

Columns include:

| Column | Meaning |
|---|---|
| `transaction_id` | Unique transaction ID |
| `store_id` | Store where transaction occurred |
| `customer_id` | Customer identifier |
| `amount` | Transaction amount |
| `category` | Product category |
| `payment_method` | Payment type |
| `timestamp` | Transaction timestamp |

This is the same type of simple dataset you will use throughout the early Spark notebooks.


In [0]:
transaction_data = [
   (1001, "Store_001", "Customer_001", 249.99, "electronics", "credit card", datetime(2026, 5, 1, 9, 15, 0)),
   (1002, "Store_001", "Customer_002", 34.50, "grocery", "cash", datetime(2026, 5, 1, 10, 5, 0)),
   (1003, "Store_002", "Customer_003", 89.99, "clothing", "debit card", datetime(2026, 5, 1, 11, 30, 0)),
   (1004, "Store_002", "Customer_001", 599.99, "electronics", "credit card", datetime(2026, 5, 2, 14, 45, 0)),
   (1005, "Store_003", "Customer_004", 22.25, "grocery", "cash", datetime(2026, 5, 2, 16, 20, 0)),
   (1006, "Store_003", "Customer_005", 129.99, "home goods", "mobile pay", datetime(2026, 6, 3, 13, 10, 0)),
   (1007, "Store_001", "Customer_006", 14.99, "office supplies", "debit card", datetime(2026, 6, 3, 17, 55, 0)),
   (1008, "Store_002", "Customer_007", 799.99, "electronics", "credit card", datetime(2026, 6, 4, 12, 0, 0)),
   (1009, "Store_003", "Customer_008", 45.00, "clothing", "cash", datetime(2026, 7, 4, 15, 35, 0)),
   (1010, "Store_001", "Customer_009", 199.99, "home goods", "credit card", datetime(2026, 7, 5, 18, 10, 0)),
]

In [0]:
transaction_schema = StructType([
   StructField("transaction_id", IntegerType(), nullable=False),
   StructField("store_id", StringType(), nullable=True),
   StructField("customer_id", StringType(), nullable=True),
   StructField("amount", DoubleType(), nullable=True),
   StructField("category", StringType(), nullable=True),
   StructField("payment_method", StringType(), nullable=True),
   StructField("timestamp", TimestampType(), nullable=True)
])

In [0]:
transactions_df = spark.createDataFrame(
   transaction_data,
   schema=transaction_schema
)

display(transactions_df)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp
1001,Store_001,Customer_001,249.99,electronics,credit card,2026-05-01T09:15:00.000Z
1002,Store_001,Customer_002,34.5,grocery,cash,2026-05-01T10:05:00.000Z
1003,Store_002,Customer_003,89.99,clothing,debit card,2026-05-01T11:30:00.000Z
1004,Store_002,Customer_001,599.99,electronics,credit card,2026-05-02T14:45:00.000Z
1005,Store_003,Customer_004,22.25,grocery,cash,2026-05-02T16:20:00.000Z
1006,Store_003,Customer_005,129.99,home goods,mobile pay,2026-06-03T13:10:00.000Z
1007,Store_001,Customer_006,14.99,office supplies,debit card,2026-06-03T17:55:00.000Z
1008,Store_002,Customer_007,799.99,electronics,credit card,2026-06-04T12:00:00.000Z
1009,Store_003,Customer_008,45.0,clothing,cash,2026-07-04T15:35:00.000Z
1010,Store_001,Customer_009,199.99,home goods,credit card,2026-07-05T18:10:00.000Z


In [0]:
transactions_df.printSchema()

root
 |-- transaction_id: integer (nullable = false)
 |-- store_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



# Section 2: Select Columns

## 2. Select Columns with `select()`

The `select()` method lets you choose which columns you want to keep.

This is one of the most common Spark operations.

Example:

```python
df.select("column_1", "column_2")
```
Selecting only needed columns is a good habit because it keeps your DataFrame focused and can improve performance in larger workflows.

In [0]:
# Select only a few columns from the transaction DataFrame.

selected_columns_df = transactions_df.select(
   "transaction_id",
   "store_id",
   "amount",
   "category"
)

display(selected_columns_df)

transaction_id,store_id,amount,category
1001,Store_001,249.99,electronics
1002,Store_001,34.5,grocery
1003,Store_002,89.99,clothing
1004,Store_002,599.99,electronics
1005,Store_003,22.25,grocery
1006,Store_003,129.99,home goods
1007,Store_001,14.99,office supplies
1008,Store_002,799.99,electronics
1009,Store_003,45.0,clothing
1010,Store_001,199.99,home goods


In [0]:
# You can also select columns using F.col().
# This is useful when applying expressions or aliases.

selected_with_col_df = transactions_df.select(
   F.col("transaction_id"),
   F.col("customer_id"),
   F.col("amount")
)

display(selected_with_col_df)

transaction_id,customer_id,amount
1001,Customer_001,249.99
1002,Customer_002,34.5
1003,Customer_003,89.99
1004,Customer_001,599.99
1005,Customer_004,22.25
1006,Customer_005,129.99
1007,Customer_006,14.99
1008,Customer_007,799.99
1009,Customer_008,45.0
1010,Customer_009,199.99


# Section 3: Filter Rows

## 3. Filter Rows with `filter()`

The `filter()` method keeps only rows that match a condition.

Example:

```python
df.filter(F.col("amount") > 100)
```
This is equivalent to a SQL WHERE clause.
You can use comparison operators such as:

```
>
Greater than

>=
Greater than or equal to

<
Less than

<=
Less than or equal to

==
Equal to

!=
Not equal to
```


In [0]:
# Filter for transactions greater than 100 dollars.

transactions_over_100_df = transactions_df.filter(
   F.col("amount") > 100
)

display(transactions_over_100_df)


transaction_id,store_id,customer_id,amount,category,payment_method,timestamp
1001,Store_001,Customer_001,249.99,electronics,credit card,2026-05-01T09:15:00.000Z
1004,Store_002,Customer_001,599.99,electronics,credit card,2026-05-02T14:45:00.000Z
1006,Store_003,Customer_005,129.99,home goods,mobile pay,2026-06-03T13:10:00.000Z
1008,Store_002,Customer_007,799.99,electronics,credit card,2026-06-04T12:00:00.000Z
1010,Store_001,Customer_009,199.99,home goods,credit card,2026-07-05T18:10:00.000Z


In [0]:
# Filter for electronics transactions.

electronics_df = transactions_df.filter(
   F.col("category") == "electronics"
)

display(electronics_df)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp
1001,Store_001,Customer_001,249.99,electronics,credit card,2026-05-01T09:15:00.000Z
1004,Store_002,Customer_001,599.99,electronics,credit card,2026-05-02T14:45:00.000Z
1008,Store_002,Customer_007,799.99,electronics,credit card,2026-06-04T12:00:00.000Z


In [0]:
# Combine multiple filter conditions.
# Use & for AND.
# Use | for OR.
# Always wrap each condition in parentheses.

high_value_electronics_df = transactions_df.filter(
   (F.col("amount") > 200) &
   (F.col("category") == "electronics")
)

display(high_value_electronics_df)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp
1001,Store_001,Customer_001,249.99,electronics,credit card,2026-05-01T09:15:00.000Z
1004,Store_002,Customer_001,599.99,electronics,credit card,2026-05-02T14:45:00.000Z
1008,Store_002,Customer_007,799.99,electronics,credit card,2026-06-04T12:00:00.000Z


# Section 4: Filter Rows with where()

## 4. Filter Rows with `where()`

`where()` works the same way as `filter()`.

These two are equivalent:

```python
df.filter(F.col("amount") > 100)
df.where(F.col("amount") > 100)
```
Some people prefer where() because it feels similar to SQL.

In [0]:
# Use where() instead of filter().
# This returns the same type of result as filter().

where_example_df = transactions_df.where(
   F.col("payment_method") == "credit card"
)

display(where_example_df)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp
1001,Store_001,Customer_001,249.99,electronics,credit card,2026-05-01T09:15:00.000Z
1004,Store_002,Customer_001,599.99,electronics,credit card,2026-05-02T14:45:00.000Z
1008,Store_002,Customer_007,799.99,electronics,credit card,2026-06-04T12:00:00.000Z
1010,Store_001,Customer_009,199.99,home goods,credit card,2026-07-05T18:10:00.000Z


# Section 5: Rename Columns

## 5. Rename Columns with `withColumnRenamed()`

The `withColumnRenamed()` method renames one column at a time.

Example:

```python
df.withColumnRenamed("old_name", "new_name")
```
Renaming columns is useful when preparing clean outputs for reports, dashboards, or downstream tables.


In [0]:
# Rename amount to transaction_amount.

renamed_df = transactions_df.withColumnRenamed(
   "amount",
   "transaction_amount"
)

display(renamed_df)

transaction_id,store_id,customer_id,transaction_amount,category,payment_method,timestamp
1001,Store_001,Customer_001,249.99,electronics,credit card,2026-05-01T09:15:00.000Z
1002,Store_001,Customer_002,34.5,grocery,cash,2026-05-01T10:05:00.000Z
1003,Store_002,Customer_003,89.99,clothing,debit card,2026-05-01T11:30:00.000Z
1004,Store_002,Customer_001,599.99,electronics,credit card,2026-05-02T14:45:00.000Z
1005,Store_003,Customer_004,22.25,grocery,cash,2026-05-02T16:20:00.000Z
1006,Store_003,Customer_005,129.99,home goods,mobile pay,2026-06-03T13:10:00.000Z
1007,Store_001,Customer_006,14.99,office supplies,debit card,2026-06-03T17:55:00.000Z
1008,Store_002,Customer_007,799.99,electronics,credit card,2026-06-04T12:00:00.000Z
1009,Store_003,Customer_008,45.0,clothing,cash,2026-07-04T15:35:00.000Z
1010,Store_001,Customer_009,199.99,home goods,credit card,2026-07-05T18:10:00.000Z


In [0]:
# Rename multiple columns by chaining withColumnRenamed().

renamed_multiple_df = (
   transactions_df
   .withColumnRenamed("amount", "transaction_amount")
   .withColumnRenamed("timestamp", "transaction_timestamp")
)

display(renamed_multiple_df)

transaction_id,store_id,customer_id,transaction_amount,category,payment_method,transaction_timestamp
1001,Store_001,Customer_001,249.99,electronics,credit card,2026-05-01T09:15:00.000Z
1002,Store_001,Customer_002,34.5,grocery,cash,2026-05-01T10:05:00.000Z
1003,Store_002,Customer_003,89.99,clothing,debit card,2026-05-01T11:30:00.000Z
1004,Store_002,Customer_001,599.99,electronics,credit card,2026-05-02T14:45:00.000Z
1005,Store_003,Customer_004,22.25,grocery,cash,2026-05-02T16:20:00.000Z
1006,Store_003,Customer_005,129.99,home goods,mobile pay,2026-06-03T13:10:00.000Z
1007,Store_001,Customer_006,14.99,office supplies,debit card,2026-06-03T17:55:00.000Z
1008,Store_002,Customer_007,799.99,electronics,credit card,2026-06-04T12:00:00.000Z
1009,Store_003,Customer_008,45.0,clothing,cash,2026-07-04T15:35:00.000Z
1010,Store_001,Customer_009,199.99,home goods,credit card,2026-07-05T18:10:00.000Z


# Section 6: Create New Columns

## 6. Create New Columns with `withColumn()`

The `withColumn()` method creates a new column or replaces an existing column.

Example:

```python
df.withColumn("new_column", expression)
```
Common uses:
```
Create flags
Create buckets
Perform arithmetic
Extract date parts
Standardize strings
```


In [0]:
# Create a new column that adds estimated sales tax.
# This is just an example. The tax rate is arbitrary.

transactions_with_tax_df = transactions_df.withColumn(
   "estimated_tax",
   F.round(F.col("amount") * 0.07, 2)
)

display(transactions_with_tax_df)


transaction_id,store_id,customer_id,amount,category,payment_method,timestamp,estimated_tax
1001,Store_001,Customer_001,249.99,electronics,credit card,2026-05-01T09:15:00.000Z,17.5
1002,Store_001,Customer_002,34.5,grocery,cash,2026-05-01T10:05:00.000Z,2.42
1003,Store_002,Customer_003,89.99,clothing,debit card,2026-05-01T11:30:00.000Z,6.3
1004,Store_002,Customer_001,599.99,electronics,credit card,2026-05-02T14:45:00.000Z,42.0
1005,Store_003,Customer_004,22.25,grocery,cash,2026-05-02T16:20:00.000Z,1.56
1006,Store_003,Customer_005,129.99,home goods,mobile pay,2026-06-03T13:10:00.000Z,9.1
1007,Store_001,Customer_006,14.99,office supplies,debit card,2026-06-03T17:55:00.000Z,1.05
1008,Store_002,Customer_007,799.99,electronics,credit card,2026-06-04T12:00:00.000Z,56.0
1009,Store_003,Customer_008,45.0,clothing,cash,2026-07-04T15:35:00.000Z,3.15
1010,Store_001,Customer_009,199.99,home goods,credit card,2026-07-05T18:10:00.000Z,14.0


In [0]:
# Create a total amount column.

transactions_with_total_df = transactions_with_tax_df.withColumn(
   "estimated_total",
   F.round(F.col("amount") + F.col("estimated_tax"), 2)
)

display(transactions_with_total_df)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp,estimated_tax,estimated_total
1001,Store_001,Customer_001,249.99,electronics,credit card,2026-05-01T09:15:00.000Z,17.5,267.49
1002,Store_001,Customer_002,34.5,grocery,cash,2026-05-01T10:05:00.000Z,2.42,36.92
1003,Store_002,Customer_003,89.99,clothing,debit card,2026-05-01T11:30:00.000Z,6.3,96.29
1004,Store_002,Customer_001,599.99,electronics,credit card,2026-05-02T14:45:00.000Z,42.0,641.99
1005,Store_003,Customer_004,22.25,grocery,cash,2026-05-02T16:20:00.000Z,1.56,23.81
1006,Store_003,Customer_005,129.99,home goods,mobile pay,2026-06-03T13:10:00.000Z,9.1,139.09
1007,Store_001,Customer_006,14.99,office supplies,debit card,2026-06-03T17:55:00.000Z,1.05,16.04
1008,Store_002,Customer_007,799.99,electronics,credit card,2026-06-04T12:00:00.000Z,56.0,855.99
1009,Store_003,Customer_008,45.0,clothing,cash,2026-07-04T15:35:00.000Z,3.15,48.15
1010,Store_001,Customer_009,199.99,home goods,credit card,2026-07-05T18:10:00.000Z,14.0,213.99


In [0]:
# Create a high-value transaction flag.
# True if amount is greater than or equal to 200.

transactions_with_flag_df = transactions_df.withColumn(
   "is_high_value",
   F.when(F.col("amount") >= 200, True).otherwise(False)
)

display(transactions_with_flag_df)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp,is_high_value
1001,Store_001,Customer_001,249.99,electronics,credit card,2026-05-01T09:15:00.000Z,true
1002,Store_001,Customer_002,34.5,grocery,cash,2026-05-01T10:05:00.000Z,false
1003,Store_002,Customer_003,89.99,clothing,debit card,2026-05-01T11:30:00.000Z,false
1004,Store_002,Customer_001,599.99,electronics,credit card,2026-05-02T14:45:00.000Z,true
1005,Store_003,Customer_004,22.25,grocery,cash,2026-05-02T16:20:00.000Z,false
1006,Store_003,Customer_005,129.99,home goods,mobile pay,2026-06-03T13:10:00.000Z,false
1007,Store_001,Customer_006,14.99,office supplies,debit card,2026-06-03T17:55:00.000Z,false
1008,Store_002,Customer_007,799.99,electronics,credit card,2026-06-04T12:00:00.000Z,true
1009,Store_003,Customer_008,45.0,clothing,cash,2026-07-04T15:35:00.000Z,false
1010,Store_001,Customer_009,199.99,home goods,credit card,2026-07-05T18:10:00.000Z,false


In [0]:
# Create an amount bucket.
# This is useful for grouped analysis later.

transactions_with_bucket_df = transactions_df.withColumn(
   "amount_bucket",
   F.when(F.col("amount") >= 500, "very_high")
    .when(F.col("amount") >= 200, "high")
    .when(F.col("amount") >= 50, "medium")
    .otherwise("low")
)

display(transactions_with_bucket_df)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp,amount_bucket
1001,Store_001,Customer_001,249.99,electronics,credit card,2026-05-01T09:15:00.000Z,high
1002,Store_001,Customer_002,34.5,grocery,cash,2026-05-01T10:05:00.000Z,low
1003,Store_002,Customer_003,89.99,clothing,debit card,2026-05-01T11:30:00.000Z,medium
1004,Store_002,Customer_001,599.99,electronics,credit card,2026-05-02T14:45:00.000Z,very_high
1005,Store_003,Customer_004,22.25,grocery,cash,2026-05-02T16:20:00.000Z,low
1006,Store_003,Customer_005,129.99,home goods,mobile pay,2026-06-03T13:10:00.000Z,medium
1007,Store_001,Customer_006,14.99,office supplies,debit card,2026-06-03T17:55:00.000Z,low
1008,Store_002,Customer_007,799.99,electronics,credit card,2026-06-04T12:00:00.000Z,very_high
1009,Store_003,Customer_008,45.0,clothing,cash,2026-07-04T15:35:00.000Z,low
1010,Store_001,Customer_009,199.99,home goods,credit card,2026-07-05T18:10:00.000Z,medium


# Section 7: Drop Columns

## 7. Drop Columns with `drop()`

The `drop()` method removes one or more columns.

Example:

```python
df.drop("column_name")
```
Dropping columns is useful when cleaning outputs or removing columns that are no longer needed.

In [0]:
# Drop the payment_method column.

dropped_column_df = transactions_df.drop("payment_method")

display(dropped_column_df)

transaction_id,store_id,customer_id,amount,category,timestamp
1001,Store_001,Customer_001,249.99,electronics,2026-05-01T09:15:00.000Z
1002,Store_001,Customer_002,34.5,grocery,2026-05-01T10:05:00.000Z
1003,Store_002,Customer_003,89.99,clothing,2026-05-01T11:30:00.000Z
1004,Store_002,Customer_001,599.99,electronics,2026-05-02T14:45:00.000Z
1005,Store_003,Customer_004,22.25,grocery,2026-05-02T16:20:00.000Z
1006,Store_003,Customer_005,129.99,home goods,2026-06-03T13:10:00.000Z
1007,Store_001,Customer_006,14.99,office supplies,2026-06-03T17:55:00.000Z
1008,Store_002,Customer_007,799.99,electronics,2026-06-04T12:00:00.000Z
1009,Store_003,Customer_008,45.0,clothing,2026-07-04T15:35:00.000Z
1010,Store_001,Customer_009,199.99,home goods,2026-07-05T18:10:00.000Z


In [0]:
# Drop multiple columns.

dropped_multiple_columns_df = transactions_df.drop(
   "payment_method",
   "timestamp"
)

display(dropped_multiple_columns_df)

transaction_id,store_id,customer_id,amount,category
1001,Store_001,Customer_001,249.99,electronics
1002,Store_001,Customer_002,34.5,grocery
1003,Store_002,Customer_003,89.99,clothing
1004,Store_002,Customer_001,599.99,electronics
1005,Store_003,Customer_004,22.25,grocery
1006,Store_003,Customer_005,129.99,home goods
1007,Store_001,Customer_006,14.99,office supplies
1008,Store_002,Customer_007,799.99,electronics
1009,Store_003,Customer_008,45.0,clothing
1010,Store_001,Customer_009,199.99,home goods


# Section 8: Sort Data

## 8. Sort Data with `orderBy()`

The `orderBy()` method sorts a DataFrame.

Examples:

```python
df.orderBy("amount")
df.orderBy(F.col("amount").desc())
```
Sorting can be expensive at scale because it may require data movement.
For small learning examples, it is fine.

In [0]:
# Sort by amount from highest to lowest.

amount_descending_df = transactions_df.orderBy(
   F.col("amount").desc()
)

display(amount_descending_df)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp
1008,Store_002,Customer_007,799.99,electronics,credit card,2026-06-04T12:00:00.000Z
1004,Store_002,Customer_001,599.99,electronics,credit card,2026-05-02T14:45:00.000Z
1001,Store_001,Customer_001,249.99,electronics,credit card,2026-05-01T09:15:00.000Z
1010,Store_001,Customer_009,199.99,home goods,credit card,2026-07-05T18:10:00.000Z
1006,Store_003,Customer_005,129.99,home goods,mobile pay,2026-06-03T13:10:00.000Z
1003,Store_002,Customer_003,89.99,clothing,debit card,2026-05-01T11:30:00.000Z
1009,Store_003,Customer_008,45.0,clothing,cash,2026-07-04T15:35:00.000Z
1002,Store_001,Customer_002,34.5,grocery,cash,2026-05-01T10:05:00.000Z
1005,Store_003,Customer_004,22.25,grocery,cash,2026-05-02T16:20:00.000Z
1007,Store_001,Customer_006,14.99,office supplies,debit card,2026-06-03T17:55:00.000Z


In [0]:
# Sort by category, then by amount descending within category.

category_amount_sorted_df = transactions_df.orderBy(
   "category",
   F.col("amount").desc()
)

display(category_amount_sorted_df)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp
1003,Store_002,Customer_003,89.99,clothing,debit card,2026-05-01T11:30:00.000Z
1009,Store_003,Customer_008,45.0,clothing,cash,2026-07-04T15:35:00.000Z
1008,Store_002,Customer_007,799.99,electronics,credit card,2026-06-04T12:00:00.000Z
1004,Store_002,Customer_001,599.99,electronics,credit card,2026-05-02T14:45:00.000Z
1001,Store_001,Customer_001,249.99,electronics,credit card,2026-05-01T09:15:00.000Z
1002,Store_001,Customer_002,34.5,grocery,cash,2026-05-01T10:05:00.000Z
1005,Store_003,Customer_004,22.25,grocery,cash,2026-05-02T16:20:00.000Z
1010,Store_001,Customer_009,199.99,home goods,credit card,2026-07-05T18:10:00.000Z
1006,Store_003,Customer_005,129.99,home goods,mobile pay,2026-06-03T13:10:00.000Z
1007,Store_001,Customer_006,14.99,office supplies,debit card,2026-06-03T17:55:00.000Z


# Section 9: Limit Rows

## 9. Limit Rows with `limit()`

The `limit()` method returns a limited number of rows.

Example:

```python
df.limit(5)
```
This is useful when previewing data.
In Databricks, you will often use:
```
display(df.limit(10))
```

In [0]:
# Show only the first 5 rows.

first_5_df = transactions_df.limit(5)

display(first_5_df)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp
1001,Store_001,Customer_001,249.99,electronics,credit card,2026-05-01T09:15:00.000Z
1002,Store_001,Customer_002,34.5,grocery,cash,2026-05-01T10:05:00.000Z
1003,Store_002,Customer_003,89.99,clothing,debit card,2026-05-01T11:30:00.000Z
1004,Store_002,Customer_001,599.99,electronics,credit card,2026-05-02T14:45:00.000Z
1005,Store_003,Customer_004,22.25,grocery,cash,2026-05-02T16:20:00.000Z


In [0]:
# Combine orderBy() and limit() to get top transactions.

top_3_transactions_df = (
   transactions_df
   .orderBy(F.col("amount").desc())
   .limit(3)
)

display(top_3_transactions_df)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp
1008,Store_002,Customer_007,799.99,electronics,credit card,2026-06-04T12:00:00.000Z
1004,Store_002,Customer_001,599.99,electronics,credit card,2026-05-02T14:45:00.000Z
1001,Store_001,Customer_001,249.99,electronics,credit card,2026-05-01T09:15:00.000Z


# Section 10: Chain Multiple Transformations

## 10. Chain Multiple Transformations

In real Spark workflows, you often chain many operations together.

Example pattern:

```python
result_df = (
   df
   .filter(...)
   .select(...)
   .withColumn(...)
   .orderBy(...)
)
```
This style is common in PySpark because it keeps the transformation logic readable.

In [0]:
# Chain several transformations together.
#
# This example:
# 1. Filters for transactions >= 100
# 2. Selects useful columns
# 3. Adds a high-value flag
# 4. Sorts by amount descending

chained_transformations_df = (
   transactions_df
   .filter(F.col("amount") >= 100)
   .select(
       "transaction_id",
       "store_id",
       "customer_id",
       "amount",
       "category",
       "timestamp"
   )
   .withColumn(
       "is_high_value",
       F.when(F.col("amount") >= 200, True).otherwise(False)
   )
   .orderBy(F.col("amount").desc())
)

display(chained_transformations_df)

transaction_id,store_id,customer_id,amount,category,timestamp,is_high_value
1008,Store_002,Customer_007,799.99,electronics,2026-06-04T12:00:00.000Z,true
1004,Store_002,Customer_001,599.99,electronics,2026-05-02T14:45:00.000Z,true
1001,Store_001,Customer_001,249.99,electronics,2026-05-01T09:15:00.000Z,true
1010,Store_001,Customer_009,199.99,home goods,2026-07-05T18:10:00.000Z,false
1006,Store_003,Customer_005,129.99,home goods,2026-06-03T13:10:00.000Z,false


# Section 11: Practice Exercises

# Practice Exercises

Try these on your own.

## Exercise 1

Select only these columns from `transactions_df`:

- `transaction_id`
- `amount`
- `category`

## Exercise 2

Filter for transactions where `category` is `"grocery"`.

## Exercise 3

Filter for transactions where `amount` is less than 50.

## Exercise 4

Rename `amount` to `transaction_amount`.

## Exercise 5

Create a new column called `is_credit_card` that is `True` when `payment_method` equals `"credit card"`.

## Exercise 6

Create a new column called `transaction_month` using the timestamp column.

Hint:

```python
F.month(F.col("timestamp"))
```

## Exercise 7
Drop the payment_method column.

## Exercise 8
Sort transactions by timestamp from earliest to latest.

## Exercise 9
Find the top 5 transactions by amount.

## Exercise 10
Create your own mini-report:
Filter to one category
Select useful columns
Rename at least one column
Add one derived column
Sort the output


# Section 12: Common Mistakes

## Mistake 1: Forgetting Parentheses Around Filter Conditions

Incorrect:

```python
df.filter(F.col("amount") > 100 & F.col("category") == "electronics")
Correct:
df.filter(
   (F.col("amount") > 100) &
   (F.col("category") == "electronics")
)
```

## Mistake 2: Using Python and Instead of &
Incorrect:
```
df.filter((F.col("amount") > 100) and (F.col("category") == "electronics"))
```
Correct:
```
df.filter((F.col("amount") > 100) & (F.col("category") == "electronics"))
```
Use:
```
& for AND
| for OR
~ for NOT
```

## Mistake 3: Thinking Transformations Run Immediately
This does not execute the full job immediately:
```
filtered_df = df.filter(F.col("amount") > 100)
```
Execution happens when you call an action:
```
filtered_df.count()
```

## Mistake 4: Renaming Columns Too Late
If a column name is confusing, rename it before building reports or downstream tables.
Clean column names make your code easier to read.

## Mistake 5: Sorting Huge Data Without a Reason
Sorting can be expensive at scale.
Use sorting intentionally, especially before displaying reports or creating top-N outputs.


# Section 13: Key Takeaways

In this notebook, you learned the core DataFrame operations you will use constantly.

The most important methods were:

| Method | Purpose |
|---|---|
| `select()` | Choose columns |
| `filter()` | Keep rows matching a condition |
| `where()` | Same as `filter()` |
| `withColumnRenamed()` | Rename a column |
| `withColumn()` | Create or replace a column |
| `drop()` | Remove columns |
| `orderBy()` | Sort rows |
| `limit()` | Return a limited number of rows |

The core workflow was:

```text
Start with transaction data
   ↓
Select useful columns
   ↓
Filter high-value transactions
   ↓
Rename columns
   ↓
Create derived fields
   ↓
Drop unnecessary columns
   ↓
Sort the report
```